<a href="https://colab.research.google.com/github/MernaAziz/my-projects/blob/main/self_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Self-Attention Score Computation for a Transformer Encoder
Sentence: "What are the symptoms of diabetes?"
"""

import numpy as np

# ── 1. Sentence & tiny embedding ────────────────────────────────────────────
sentence = ["What", "are", "the", "symptoms", "of", "diabetes", "?"]
n_tokens = len(sentence)

# Embedding dimension and head dimension
d_model = 8      # keep small for readability
d_k     = 4      # dimension of Q and K vectors

np.random.seed(42)  # reproducible

# Simulate token embeddings  (n_tokens × d_model)
X = np.random.randn(n_tokens, d_model)

# ── 2. Learned weight matrices W_Q, W_K, W_V ────────────────────────────────
W_Q = np.random.randn(d_model, d_k)
W_K = np.random.randn(d_model, d_k)
W_V = np.random.randn(d_model, d_k)

# ── 3. Compute Q, K, V ──────────────────────────────────────────────────────
Q = X @ W_Q   # (n_tokens × d_k)
K = X @ W_K   # (n_tokens × d_k)
V = X @ W_V   # (n_tokens × d_k)

# ── 4. Scaled dot-product attention scores ──────────────────────────────────
#   score(Q, K) = (Q · K^T) / sqrt(d_k)
raw_scores = Q @ K.T / np.sqrt(d_k)   # (n_tokens × n_tokens)

# ── 5. Softmax over each row (each query token normalised over all keys) ─────
def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))  # numerical stability
    return e / e.sum(axis=-1, keepdims=True)

attention_weights = softmax(raw_scores)   # (n_tokens × n_tokens)

# ── 6. Weighted sum of Values ────────────────────────────────────────────────
output = attention_weights @ V   # (n_tokens × d_k)

# ── 7. Pretty-print results ──────────────────────────────────────────────────
print("=" * 65)
print("SELF-ATTENTION DEMONSTRATION")
print("Sentence:", " ".join(sentence))
print("=" * 65)

print("\n── Raw attention scores (before softmax) ──")
header = f"{'':12s}" + "".join(f"{w:>10s}" for w in sentence)
print(header)
for i, q_word in enumerate(sentence):
    row = f"{q_word:<12s}" + "".join(f"{raw_scores[i,j]:>10.3f}" for j in range(n_tokens))
    print(row)

print("\n── Attention weights (after softmax, rows sum to 1.0) ──")
print(header)
for i, q_word in enumerate(sentence):
    row = f"{q_word:<12s}" + "".join(f"{attention_weights[i,j]:>10.4f}" for j in range(n_tokens))
    print(row)

print("\n── Most attended token for each query word ──")
for i, q_word in enumerate(sentence):
    best_j = np.argmax(attention_weights[i])
    print(f"  '{q_word}' attends most to '{sentence[best_j]}'  "
          f"(weight = {attention_weights[i, best_j]:.4f})")

print("\n── Output vectors (contextual representations) shape:", output.shape, "──")
print("  (each token now encodes information from the whole sentence)")
print("=" * 65)

SELF-ATTENTION DEMONSTRATION
Sentence: What are the symptoms of diabetes ?

── Raw attention scores (before softmax) ──
                  What       are       the  symptoms        of  diabetes         ?
What             0.700     6.234     1.948     4.741    15.904     2.370     4.496
are              0.205   -11.452    -4.807    -3.791   -10.526     0.970     7.204
the              2.575   -14.505    -0.396   -12.305   -23.299    -5.703    -0.634
symptoms        -3.025    11.352    -3.403    10.706     8.128     4.532    -6.949
of               3.296   -12.077    -0.604    -6.261    -1.590    -0.203    14.069
diabetes        -1.946    12.686    -1.438    10.654    13.445     4.288    -4.864
?                0.170    12.027     4.793     5.482    17.141     0.644    -2.543

── Attention weights (after softmax, rows sum to 1.0) ──
                  What       are       the  symptoms        of  diabetes         ?
What            0.0000    0.0001    0.0000    0.0000    0.9999    0.0000   